[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/11_random_forest/02_random_forest_from_scratch.ipynb)

# Bài 02 — Random Forest from scratch

**🎯 Mục tiêu**
- Hiểu vấn đề của bagging: các cây **tương quan** (cùng chọn feature mạnh nhất ở gốc).
- Random Forest = bagging + **chọn ngẫu nhiên tập feature ở mỗi split** (`max_features`) → giảm tương quan → giảm variance.
- Cài đặt Random Forest (phân loại + hồi quy); đối chiếu với `sklearn.ensemble.RandomForestClassifier`.
- Thấy tác động của `max_features` lên tương quan giữa cây và accuracy.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/11_random_forest

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
churn = make_customer_churn(n=5000, seed=11)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
y = churn["churn"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X.to_numpy(), y, test_size=0.3, random_state=0, stratify=y)
feature_names = list(X.columns)
print(X_train.shape)

## 1. Vấn đề: các cây bagging giống nhau

Nếu có một feature rất mạnh, **gần như mọi** cây bootstrap đều chọn nó ở gốc → các cây có cấu trúc giống nhau → dự đoán **tương quan cao** → theo công thức bài 01, variance của trung bình không giảm được nhiều.

In [ ]:
def root_features(max_features, n_trees=50):
    roots = []
    for b in range(n_trees):
        idx = np.random.default_rng(b).integers(0, len(X_train), len(X_train))
        t = DecisionTreeClassifier(max_features=max_features, random_state=b).fit(X_train[idx], y_train[idx])
        roots.append(feature_names[t.tree_.feature[0]])
    return pd.Series(roots).value_counts()


print("Bagging (max_features = tất cả) — feature ở gốc của 50 cây:\n", root_features(None).to_string(), "\n")
print("Random Forest (max_features = 'sqrt') — feature ở gốc của 50 cây:\n", root_features("sqrt").to_string())

## 2. Ý tưởng Random Forest (Breiman, 2001)

Ở **mỗi split** của mỗi cây, chỉ xét một tập con ngẫu nhiên $m$ trong $d$ feature (thường $m = \sqrt d$ cho phân loại, $m = d$ hoặc $d/3$ cho hồi quy). Feature mạnh nhất không phải lúc nào cũng có mặt → các cây buộc phải khám phá cấu trúc khác → **đa dạng** hơn.

Đổi lại, mỗi cây đơn lẻ **kém hơn** một chút (bias tăng nhẹ) — nhưng trung bình của chúng tốt hơn vì tương quan $\rho$ giảm mạnh.

In [ ]:
class RandomForest:
    def __init__(self, n_estimators=100, max_features="sqrt", max_depth=None, min_samples_leaf=1,
                 task="classification", seed=0):
        self.n_estimators, self.max_features, self.task, self.seed = n_estimators, max_features, task, seed
        self.max_depth, self.min_samples_leaf = max_depth, min_samples_leaf

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        Tree = DecisionTreeClassifier if self.task == "classification" else DecisionTreeRegressor
        self.trees_, self.bootstrap_idx_ = [], []
        for _ in range(self.n_estimators):
            idx = rng.integers(0, len(X), len(X))                                 # 1. bootstrap
            tree = Tree(max_features=self.max_features,                          # 2. feature ngẫu nhiên MỖI SPLIT
                        max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf,
                        random_state=int(rng.integers(1 << 31)))
            self.trees_.append(tree.fit(X[idx], y[idx]))
            self.bootstrap_idx_.append(idx)
        if self.task == "classification":
            self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X):
        return np.mean([t.predict_proba(X) for t in self.trees_], axis=0)       # 3. trung bình xác suất

    def predict(self, X):
        if self.task == "classification":
            return self.classes_[np.argmax(self.predict_proba(X), axis=1)]
        return np.mean([t.predict(X) for t in self.trees_], axis=0)

> 📝 Ở đây ta dùng cây của scikit-learn làm "viên gạch" (đã tự cài cây ở chương 10) và tự viết phần **bootstrap + random feature + tổng hợp**. Phiên bản dùng cây tự viết hoàn toàn: `ml_course.algorithms.RandomForestScratch`. Lưu ý: tham số `max_features` của cây chọn feature ngẫu nhiên **ở mỗi split** — không phải một lần cho cả cây (đó là "random subspace", kém hiệu quả hơn).

In [ ]:
start = time.perf_counter()
ours = RandomForest(n_estimators=200, min_samples_leaf=3, seed=0).fit(X_train, y_train)
t_ours = time.perf_counter() - start
start = time.perf_counter()
sk = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=0, n_jobs=-1).fit(X_train, y_train)
t_sk = time.perf_counter() - start
single = DecisionTreeClassifier(min_samples_leaf=3, random_state=0).fit(X_train, y_train)
for name, model in [("1 cây", single), ("RF scratch", ours), ("RF sklearn", sk)]:
    p = model.predict_proba(X_test)[:, 1]
    print(f"{name:11s} accuracy = {np.mean(model.predict(X_test) == y_test):.3f} | ROC-AUC = {roc_auc_score(y_test, p):.4f}")
print(f"thời gian train: scratch {t_ours:.1f}s | sklearn (n_jobs=-1) {t_sk:.1f}s")

## 3. `max_features`: đánh đổi giữa sức mạnh từng cây và tương quan

Để thấy rõ đánh đổi, dùng dữ liệu tổng hợp 20 feature (5 có ích, 3 dư thừa — tổ hợp tuyến tính của feature có ích, 12 nhiễu; 5% nhãn bị lật). Với mỗi `max_features`, đo (a) **tương quan trung bình** giữa xác suất dự đoán của các cặp cây trên tập test, (b) ROC-AUC trung bình của **từng cây**, (c) ROC-AUC của **cả rừng**:

In [ ]:
from sklearn.datasets import make_classification

Xs, ys = make_classification(n_samples=4000, n_features=20, n_informative=5, n_redundant=3, flip_y=0.05, random_state=0)
Xs_tr, Xs_te, ys_tr, ys_te = Xs[:2000], Xs[2000:], ys[:2000], ys[2000:]
rows = []
for m in [1, 2, 4, 7, 10, 20]:
    rf = RandomForest(n_estimators=60, max_features=m, min_samples_leaf=3, seed=1).fit(Xs_tr, ys_tr)
    P = np.array([t.predict_proba(Xs_te)[:, 1] for t in rf.trees_])
    corr = np.corrcoef(P)
    rows.append({"max_features": m, "tương quan giữa cây": corr[np.triu_indices_from(corr, k=1)].mean(),
                 "AUC trung bình 1 cây": np.mean([roc_auc_score(ys_te, p) for p in P]),
                 "AUC cả rừng": roc_auc_score(ys_te, P.mean(0))})
res = pd.DataFrame(rows).round(4)
res

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(res["max_features"], res["tương quan giữa cây"], "o-", color="tab:orange")
axes[0].set_title("Tương quan giữa các cây")
axes[1].plot(res["max_features"], res["AUC trung bình 1 cây"], "^--", label="1 cây (trung bình)")
axes[1].plot(res["max_features"], res["AUC cả rừng"], "s-", label="cả rừng")
axes[1].set_title("ROC-AUC test")
axes[1].legend()
for ax in axes:
    ax.set_xlabel("max_features (20 = bagging)")
plt.show()

Đọc bảng:
- `max_features` tăng → **từng cây mạnh hơn** (thấy được feature tốt nhất thường xuyên hơn) nhưng **tương quan giữa các cây tăng** mạnh.
- `max_features = 1`: cây rất đa dạng nhưng quá yếu; `max_features = 20` (bagging): cây mạnh nhưng giống nhau. Rừng tốt nhất nằm **ở giữa**.
- Đây chính là công thức $\rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$ của bài 01 trong thực tế. `max_features` là hyperparameter quan trọng nhất của Random Forest — chọn bằng CV hoặc OOB (bài 03).

Trên dữ liệu churn (nhãn nhiễu nhiều, vài feature mạnh), giá trị tốt nhất thường nhỏ hơn mặc định:

In [ ]:
for m in [1, 2, 4, "sqrt", 8, None]:
    auc = roc_auc_score(y_test, RandomForestClassifier(n_estimators=200, max_features=m, min_samples_leaf=3, random_state=0)
                        .fit(X_train, y_train).predict_proba(X_test)[:, 1])
    print(f"churn, max_features = {str(m):5s}: ROC-AUC test = {auc:.4f}")

## 4. Random Forest cho hồi quy

In [ ]:
houses = pd.read_csv("../../data/sample/house_prices.csv")
Xh = pd.get_dummies(houses.drop(columns=["price_billion_vnd"]), columns=["district"], dtype=float).to_numpy()
yh = houses["price_billion_vnd"].to_numpy()
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.25, random_state=0)
for name, model in [("1 cây", DecisionTreeRegressor(random_state=0)),
                    ("RF scratch", RandomForest(n_estimators=200, max_features=0.5, task="regression")),
                    ("RF sklearn", RandomForestRegressor(n_estimators=200, max_features=0.5, random_state=0))]:
    pred = model.fit(Xh_tr, yh_tr).predict(Xh_te)
    print(f"{name:11s} MAE test = {np.mean(np.abs(pred - yh_te)):.3f} tỉ")

## 5. Phiên bản tham chiếu (cây tự viết)

In [ ]:
from ml_course.algorithms import RandomForestScratch

small = slice(0, 1200)                         # cây thuần NumPy chậm hơn → dùng tập nhỏ
ref = RandomForestScratch(n_estimators=30, max_features="sqrt", min_samples_leaf=3, seed=0).fit(X_train[small], y_train[small])
print(f"RandomForestScratch (30 cây, 1 200 mẫu): ROC-AUC test = {roc_auc_score(y_test, ref.predict_proba(X_test)[:, 1]):.4f}")

## 🧠 Kiểm tra nhanh

1. Random Forest khác bagging cây quyết định ở điểm nào?
2. Vì sao chọn feature ngẫu nhiên **ở mỗi split** chứ không phải một lần cho cả cây?
3. `max_features = 1` thì mỗi split hoạt động thế nào? Vì sao rừng vẫn có thể tốt?
4. Mặc định `max_features` của sklearn cho phân loại và hồi quy là gì?

## 📌 Tóm tắt

- RF = bootstrap + random feature subset mỗi split + trung bình.
- `max_features` đánh đổi sức mạnh từng cây ↔ tương quan giữa cây.
- RF tốt hơn hẳn một cây và thường tốt hơn bagging thuần.

## 📚 Tài liệu tham khảo

- Breiman (2001) — *Random Forests*
- ESL chương 15
- Ho (1998) — *The Random Subspace Method*